# ♻️ Edge-AI Smart Dustbin: waste model verification

<a href="https://colab.research.google.com/github/ruta54-create/final-project/blob/main/ml/verify_model_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook checks the waste classifier **independently of the project PC**, on Google's computers:

| Step | What is verified |
|---|---|
| 1 | **Model health**: int8 file, input size, labels, size for the ESP32-S3 |
| 2 | **ESP32 compatibility**: only operations supported by TensorFlow Lite Micro |
| 3 | **Quick test** on 8 sample photos |
| 4 | **Accuracy on 709 test photos never used in training** (material + compartment, confusion matrix) |
| 5 | **The mistakes**, shown as photos |
| 6 | **Robustness**: darker, brighter, blurred, rotated, low-quality photos |
| 7 | **Speed** |
| 8 | **Verdict**: PASS / WARN / FAIL |
| 9 | **Try your own photos**: upload or webcam |

**How to run:** *Runtime → Run all* (about 3 minutes, no GPU needed). The model is downloaded from
GitHub: it is the **same int8 file** (`ml/output/model_int8.tflite`) that goes on the ESP32-S3-CAM.

Classes → compartments: **organic + paper → ORGANIC**, **plastic + metal → INORGANIC**.
If the AI is less than 70 % sure, the item goes to INORGANIC (safety rule).

## 0. Setup: download the project and the model

In [ ]:
import io, json, os, subprocess, sys, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ruta54-create/final-project.git"
REPO = Path(os.environ.get("SMARTBIN_REPO", "/content/final-project"))
if not (REPO / "ml" / "output" / "model_int8.tflite").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)

ML = REPO / "ml"
MODEL_PATH = ML / "output" / "model_int8.tflite"
LABELS = (ML / "output" / "labels.txt").read_text().split()
CARD = json.loads((ML / "output" / "metrics.json").read_text())   # numbers claimed in the model card
TEST_DIR = ML / "test_set"
SAMPLES_DIR = REPO / "frontend" / "public" / "samples"
BIN_OF = {"organic": "organic", "paper": "organic", "plastic": "inorganic", "metal": "inorganic"}
MIN_CONFIDENCE = 0.70
RESULTS = []          # (check, passed, detail) collected for the final verdict

def record(name, passed, detail="", warning_only=False):
    """warning_only=True: a known limit worth knowing about, not a failure of the model."""
    status = "PASS" if passed else ("WARN" if warning_only else "FAIL")
    RESULTS.append((name, status, detail))
    print(f"{status}  {name}" + (f"  ->  {detail}" if detail else ""))

print("model:", MODEL_PATH, f"({MODEL_PATH.stat().st_size / 1024:.0f} KB)")
print("labels:", LABELS)
print("model card says: material", CARD["int8"]["accuracy"], "| compartment", CARD["int8"]["compartment_accuracy"])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageEnhance, ImageFilter, ImageOps

try:                                            # new name of the TensorFlow Lite runtime
    from ai_edge_litert.interpreter import Interpreter, OpResolverType
except ImportError:
    import tensorflow as tf
    Interpreter, OpResolverType = tf.lite.Interpreter, tf.lite.experimental.OpResolverType

interp = Interpreter(model_path=str(MODEL_PATH), num_threads=2)
interp.allocate_tensors()
INP = interp.get_input_details()[0]
OUTP = interp.get_output_details()[0]
SIZE = int(INP["shape"][1])

def preprocess(img):
    """Exactly what the ESP32-CAM does: centre-crop to a square, resize, int8."""
    img = ImageOps.exif_transpose(img).convert("RGB")
    img = ImageOps.fit(img, (SIZE, SIZE), Image.BILINEAR, centering=(0.5, 0.5))
    scale, zero = INP["quantization"]
    x = np.asarray(img, dtype=np.float32)[None]
    return np.clip(np.round(x / scale + zero), -128, 127).astype(np.int8)

def classify(img):
    """Returns (label, confidence, {label: probability})."""
    interp.set_tensor(INP["index"], preprocess(img))
    interp.invoke()
    raw = interp.get_tensor(OUTP["index"])[0].astype(np.float32)
    scale, zero = OUTP["quantization"]
    p = (raw - zero) * scale
    p = p / max(float(p.sum()), 1e-6)
    i = int(np.argmax(p))
    return LABELS[i], float(p[i]), dict(zip(LABELS, map(float, p)))

def decide(label, conf):
    """The bin's sorting rule."""
    return "inorganic" if conf < MIN_CONFIDENCE else BIN_OF[label]

print(f"model input {SIZE}x{SIZE}, ready")

## 1. Model health

In [ ]:
size_kb = MODEL_PATH.stat().st_size / 1024
record("model file smaller than 1 MB (fits ESP32-S3 flash easily)", size_kb < 1024, f"{size_kb:.0f} KB")
record("input is int8, shape 1×S×S×3", INP["dtype"] == np.int8 and list(INP["shape"]) == [1, SIZE, SIZE, 3],
       f"{INP['dtype'].__name__} {INP['shape'].tolist()}")
record("output is int8 with one score per class", OUTP["dtype"] == np.int8 and int(OUTP["shape"][-1]) == len(LABELS),
       f"{OUTP['dtype'].__name__} {OUTP['shape'].tolist()}")
record("labels are organic, paper, plastic, metal (this order)", LABELS == ["organic", "paper", "plastic", "metal"], ", ".join(LABELS))
record("input quantisation set (raw pixels 0..255 go in)", INP["quantization"][0] > 0,
       f"scale={INP['quantization'][0]:.4f}, zero_point={INP['quantization'][1]}")

## 2. ESP32 compatibility (TensorFlow Lite Micro)

In [ ]:
# Operations that TensorFlow Lite Micro (used on the ESP32 by Edge Impulse) can run in int8
TFLM_INT8_OPS = {"ADD", "AVERAGE_POOL_2D", "CONCATENATION", "CONV_2D", "DEPTHWISE_CONV_2D", "DEQUANTIZE",
                 "FULLY_CONNECTED", "HARD_SWISH", "LOGISTIC", "MAX_POOL_2D", "MEAN", "MUL", "PAD", "QUANTIZE",
                 "RELU", "RELU6", "RESHAPE", "SOFTMAX", "SUB", "TANH"}
from collections import Counter
# A second copy WITHOUT the PC speed-up layer (XNNPACK), so we list the operations stored in the file itself
plain = Interpreter(model_path=str(MODEL_PATH), experimental_op_resolver_type=OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES)
plain.allocate_tensors()
ops = Counter(op["op_name"] for op in plain._get_ops_details())
print("operations in the model:", dict(ops))
unsupported = sorted(set(ops) - TFLM_INT8_OPS)
record("all operations supported by TensorFlow Lite Micro", not unsupported, "unsupported: " + ", ".join(unsupported) if unsupported else f"{sum(ops.values())} ops, {len(ops)} kinds")

dtypes = Counter(np.dtype(t["dtype"]).name for t in plain.get_tensor_details())
record("no float32 tensors (fully integer model)", "float32" not in dtypes, str(dict(dtypes)))
params = sum(int(np.prod(t["shape"])) for t in plain.get_tensor_details() if t["name"] and len(t["shape"]) > 1 and np.dtype(t["dtype"]) == np.int8)
print(f"≈ {params:,} int8 weights")

## 3. Quick test: 8 sample photos

In [ ]:
samples = sorted(SAMPLES_DIR.glob("*.jpg"))
fig, axes = plt.subplots(2, 4, figsize=(14, 7.5))
ok = 0
for ax, f in zip(axes.flat, samples):
    truth = f.stem.split("-")[0]
    img = Image.open(f)
    label, conf, _ = classify(img)
    ok += label == truth
    ax.imshow(ImageOps.fit(img.convert("RGB"), (SIZE, SIZE)))
    ax.set_title(f"real: {truth}\nAI: {label} {conf:.0%} → {decide(label, conf)}",
                 color="green" if label == truth else "red", fontsize=11)
    ax.axis("off")
plt.suptitle("Sample photos (never used in training)", fontsize=14)
plt.tight_layout(); plt.show()
print(f"{ok}/{len(samples)} correct")

## 4. Accuracy on the 709 test photos (never used in training)

In [ ]:
y_true, y_pred, confs, files = [], [], [], []
t0 = time.time()
for label in LABELS:
    for f in sorted((TEST_DIR / label).glob("*.jpg")):
        pred, conf, _ = classify(Image.open(f))
        y_true.append(label); y_pred.append(pred); confs.append(conf); files.append(f)
n = len(y_true)
acc = np.mean([t == p for t, p in zip(y_true, y_pred)])
bin_acc = np.mean([BIN_OF[t] == BIN_OF[p] for t, p in zip(y_true, y_pred)])
# what physically happens in the bin, with the 70 % safety rule
sorted_ok = np.mean([BIN_OF[t] == decide(p, c) for t, p, c in zip(y_true, y_pred, confs)])
compost_contamination = np.mean([BIN_OF[t] == "inorganic" and decide(p, c) == "organic" for t, p, c in zip(y_true, y_pred, confs)])
print(f"{n} photos in {time.time() - t0:.1f} s")
print(f"material correct          : {acc:.1%}   (model card: {CARD['int8']['accuracy']:.1%})")
print(f"right compartment (AI)    : {bin_acc:.1%}   (model card: {CARD['int8']['compartment_accuracy']:.1%})")
print(f"right compartment (+rule) : {sorted_ok:.1%}   (unsure items forced to inorganic)")
print(f"plastic/metal in compost  : {compost_contamination:.1%}   (the mistake that matters most)")

record("709 test photos found", n == 709, str(n))
record("material accuracy ≥ 80 %", acc >= 0.80, f"{acc:.1%}")
record("right compartment ≥ 88 %", bin_acc >= 0.88, f"{bin_acc:.1%}")
record("matches the model card (±3 points)", abs(acc - CARD["int8"]["accuracy"]) <= 0.03, f"{acc:.1%} vs {CARD['int8']['accuracy']:.1%}")
record("inorganic waste in the compost ≤ 5 % (with safety rule)", compost_contamination <= 0.05, f"{compost_contamination:.1%}")

In [ ]:
idx = {l: i for i, l in enumerate(LABELS)}
cm = np.zeros((4, 4), dtype=int)
for t, p in zip(y_true, y_pred):
    cm[idx[t], idx[p]] += 1

print(f"{'class':<9}{'precision':>10}{'recall':>8}{'F1':>7}{'photos':>8}")
for i, l in enumerate(LABELS):
    prec = cm[i, i] / max(cm[:, i].sum(), 1); rec = cm[i, i] / max(cm[i].sum(), 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-9)
    print(f"{l:<9}{prec:>10.2f}{rec:>8.2f}{f1:>7.2f}{cm[i].sum():>8}")

fig, ax = plt.subplots(figsize=(6, 5.2))
ax.imshow(cm, cmap="Greens")
ax.set_xticks(range(4), LABELS); ax.set_yticks(range(4), LABELS)
ax.set_xlabel("AI prediction"); ax.set_ylabel("real material")
ax.set_title(f"Confusion matrix, 709 test photos ({acc:.1%})")
for i in range(4):
    for j in range(4):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.tight_layout(); plt.show()
print("Plastic <-> metal mistakes are harmless (same compartment). Paper <-> plastic/metal mistakes cross compartments.")

## 5. The mistakes

In [ ]:
wrong = [(c, f, t, p) for t, p, c, f in zip(y_true, y_pred, confs, files) if t != p]
wrong.sort(key=lambda w: -w[0])                     # most confident mistakes first
print(f"{len(wrong)} mistakes out of {n}")
fig, axes = plt.subplots(3, 6, figsize=(16, 9))
for ax, (c, f, t, p) in zip(axes.flat, wrong):
    ax.imshow(Image.open(f)); ax.axis("off")
    cross = BIN_OF[t] != BIN_OF[p]
    ax.set_title(f"real {t}\nAI {p} {c:.0%}" + ("\nWRONG BIN" if cross else ""), fontsize=9.5, color="red" if cross else "darkorange")
for ax in list(axes.flat)[len(wrong):]:
    ax.axis("off")
plt.suptitle("Most confident mistakes (red = item would go to the wrong compartment)", fontsize=13)
plt.tight_layout(); plt.show()

## 6. Robustness: photos like a real camera might take

In [ ]:
def zoom_out(img):            # item smaller in the picture (camera higher above the tray)
    img = img.convert("RGB"); w, h = img.size
    bg = Image.new("RGB", (w, h), (200, 200, 200))
    bg.paste(img.resize((int(w * 0.7), int(h * 0.7))), (int(w * 0.15), int(h * 0.15)))
    return bg

def jpeg(img, q):
    buf = io.BytesIO(); img.convert("RGB").save(buf, "JPEG", quality=q); buf.seek(0)
    return Image.open(buf)

CONDITIONS = {
    "original": lambda im: im,
    "darker (60 % light)": lambda im: ImageEnhance.Brightness(im).enhance(0.6),
    "brighter (140 %)": lambda im: ImageEnhance.Brightness(im).enhance(1.4),
    "low contrast": lambda im: ImageEnhance.Contrast(im).enhance(0.6),
    "slightly blurred": lambda im: im.filter(ImageFilter.GaussianBlur(1.2)),
    "rotated 20°": lambda im: im.rotate(20, resample=Image.BILINEAR, fillcolor=(200, 200, 200)),
    "low JPEG quality (q=25)": lambda im: jpeg(im, 25),
    "smaller in picture": zoom_out,
}
rows = []
for name, fn in CONDITIONS.items():
    preds = [classify(fn(Image.open(f).convert("RGB")))[0] for f in files]
    a = np.mean([t == p for t, p in zip(y_true, preds)])
    b = np.mean([BIN_OF[t] == BIN_OF[p] for t, p in zip(y_true, preds)])
    rows.append((name, a, b))
    print(f"{name:<26} material {a:6.1%}   compartment {b:6.1%}   ({(a - rows[0][1]) * 100:+.1f} points)")
camera = [r for r in rows[1:] if r[0] != "smaller in picture"]
worst = min(camera, key=lambda r: r[2])
record("light / blur / rotation / JPEG: compartment accuracy stays ≥ 80 %", worst[2] >= 0.80, f"worst: {worst[0]} {worst[2]:.1%}")
small = rows[-1][2]
record("item small in the picture: compartment accuracy ≥ 80 %", small >= 0.80, f"{small:.1%}", warning_only=True)
if small < 0.80:
    print("   → Mount the camera so ONE item fills most of the picture (about 15-20 cm above the tray).")

## 7. Speed

In [ ]:
x = preprocess(Image.open(files[0]))
interp.set_tensor(INP["index"], x)
for _ in range(5):
    interp.invoke()
t0 = time.perf_counter()
for _ in range(200):
    interp.invoke()
ms = (time.perf_counter() - t0) / 200 * 1000
print(f"{ms:.2f} ms per photo on this computer ({1000 / ms:.0f} photos/second)")
print("On the ESP32-S3 expect roughly 0.3-0.7 s per photo (Edge Impulse shows the exact estimate).")
record("fast enough on a PC (< 50 ms)", ms < 50, f"{ms:.1f} ms")

## 8. Verdict

In [ ]:
ICON = {"PASS": "✅", "WARN": "⚠️", "FAIL": "❌"}
for name, status, detail in RESULTS:
    print(f"{ICON[status]}  {name}" + (f"   [{detail}]" if detail else ""))
count = Counter(r[1] for r in RESULTS)
print(f"\n{count['PASS']} passed, {count['WARN']} warnings, {count['FAIL']} failed (out of {len(RESULTS)})")
if count["FAIL"]:
    print("❌ MODEL NOT VERIFIED: see the ❌ lines above.")
elif count["WARN"]:
    print("✅ MODEL VERIFIED: working well. ⚠️ Read the warnings: they tell you how to build the bin so the AI stays accurate.")
else:
    print("✅ MODEL VERIFIED: working well.")

## 9. Try your own photos (optional)

* **Upload:** run the next cell and choose one or more photos of waste (one item per photo, centred).
* **Webcam:** run the cell after it, allow the camera, hold an item in front of it, press **Capture**.

Remember: the model was trained on public datasets. Photos taken inside the real bin (same light, same
tray) will work best; that's what the dashboard's *teach the AI* loop is for.

In [ ]:
def show_result(img, title=""):
    label, conf, scores = classify(img)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4))
    a1.imshow(ImageOps.fit(img.convert("RGB"), (SIZE, SIZE))); a1.axis("off")
    a1.set_title(f"{title}\nwhat the AI sees ({SIZE}×{SIZE})", fontsize=10)
    a2.barh(LABELS[::-1], [scores[l] for l in LABELS[::-1]],
            color=["#2563eb" if BIN_OF[l] == "inorganic" else "#16a34a" for l in LABELS[::-1]])
    a2.set_xlim(0, 1)
    a2.set_title(f"{label} {conf:.0%} → {decide(label, conf).upper()} compartment"
                 + (" (unsure: safety rule)" if conf < MIN_CONFIDENCE else ""))
    plt.tight_layout(); plt.show()

if IN_COLAB:
    from google.colab import files as colab_files
    uploaded = colab_files.upload()
    for name, data in uploaded.items():
        show_result(Image.open(io.BytesIO(data)), name)
else:
    folder = Path(os.environ.get("MY_PHOTOS", SAMPLES_DIR))      # outside Colab: a folder of photos
    for f in sorted(folder.glob("*.jpg"))[:2]:
        show_result(Image.open(f), f.name)

In [ ]:
# Webcam (Google Colab only): allow the camera, hold ONE item in front of it, press "Capture".
if IN_COLAB:
    from base64 import b64decode
    from IPython.display import Javascript, display
    from google.colab.output import eval_js

    def take_photo(quality=0.9):
        display(Javascript("""
        async function takePhoto(quality) {
          const div = document.createElement('div');
          const button = document.createElement('button');
          button.textContent = 'Capture';
          div.appendChild(button);
          const video = document.createElement('video');
          video.style.display = 'block';
          const stream = await navigator.mediaDevices.getUserMedia({video: true});
          document.body.appendChild(div);
          div.appendChild(video);
          video.srcObject = stream;
          await video.play();
          google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
          await new Promise((resolve) => button.onclick = resolve);
          const canvas = document.createElement('canvas');
          canvas.width = video.videoWidth;
          canvas.height = video.videoHeight;
          canvas.getContext('2d').drawImage(video, 0, 0);
          stream.getVideoTracks()[0].stop();
          div.remove();
          return canvas.toDataURL('image/jpeg', quality);
        }"""))
        data = eval_js(f"takePhoto({quality})")
        return Image.open(io.BytesIO(b64decode(data.split(",")[1])))

    show_result(take_photo(), "webcam photo")
else:
    print("Webcam capture works inside Google Colab.")